# NB1: OT-centric Meta-Feature Extraction

10개 데이터셋에서 OT(target) 채널 단일 시리즈 기반 메타피처(A/B/C/E, ~43개) + OT-vs-Others 관계 메타피처(D, 10개) 추출.

**입력**: `../dataset/*.csv` (10개)

**출력** (`outputs/nb1/`):
- `dataset_metadata.csv`: dataset → (T, C, ot_idx, ot_name)
- `meta_features_full.csv`: (10, ~53) wide
- `meta_features_sample_long.csv`: 10 × 50 repeats × features (long)
- `meta_features_sample_summary.csv`: sample 통계 요약
- `representativeness_report.csv`: per (dataset, feature) representativeness
- `feature_groups.csv`: feature → (group, ot_relation)


## Section 0 — Setup

In [ ]:
# !pip install numpy pandas scipy scikit-learn statsmodels pycatch22 antropy nolds PyWavelets EntropyHub xgboost lightgbm shap matplotlib seaborn tqdm

In [ ]:
from __future__ import annotations
from pathlib import Path
from typing import Optional
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

np.random.seed(42)
OUT_DIR = Path('outputs/nb1')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output dir:', OUT_DIR.resolve())

## Section 1 — 데이터 로딩 + OT 식별

OT 식별은 컬럼 이름 우선('OT' 존재 시), 없으면 마지막 컬럼.
실측: ETTh*/ETTm*/Exchange/Weather/ILI/Milano-6165 → 'OT' 컬럼명 매칭, Traffic/Electricity → 숫자 컬럼이라 fallback.

In [ ]:
# ===== USER CONFIG =====
# 다른 Milano 지역 추가 시 DATASET_PATHS 와 PERIOD_HINTS 두 dict 모두에 명시
DATASET_PATHS = {
    'ETTh1':       '../dataset/ETTh1.csv',
    'ETTh2':       '../dataset/ETTh2.csv',
    'ETTm1':       '../dataset/ETTm1.csv',
    'ETTm2':       '../dataset/ETTm2.csv',
    'Traffic':     '../dataset/traffic.csv',
    'Electricity': '../dataset/electricity.csv',
    'Exchange':    '../dataset/exchange_rate.csv',
    'Weather':     '../dataset/weather.csv',
    'ILI':         '../dataset/national_illness.csv',
    'Milano-6165': '../dataset/milano_6165.csv',
    'Milano-8815': '../dataset/milano_add/milano_8815.csv',
    'Milano-410': '../dataset/milano_add/milano_410.csv',
    'Milano-625': '../dataset/milano_add/milano_625.csv',
    'Milano-821': '../dataset/milano_add/milano_821.csv',
    'Milano-2943': '../dataset/milano_add/milano_2943.csv',
    'Milano-4641': '../dataset/milano_add/milano_4641.csv',
    'Milano-5163': '../dataset/milano_add/milano_5163.csv',
    'Milano-5827': '../dataset/milano_add/milano_5827.csv',
    'Milano-6400': '../dataset/milano_add/milano_6400.csv',
    'Milano-6610': '../dataset/milano_add/milano_6610.csv',
    'Milano-6710': '../dataset/milano_add/milano_6710.csv',
    'Milano-6797': '../dataset/milano_add/milano_6797.csv',
    'Milano-7087': '../dataset/milano_add/milano_7087.csv',
    'Milano-7108': '../dataset/milano_add/milano_7108.csv',
    'Milano-7315': '../dataset/milano_add/milano_7315.csv',
    'Milano-8284': '../dataset/milano_add/milano_8284.csv',
    'Milano-8293': '../dataset/milano_add/milano_8293.csv',
    'Milano-8440': '../dataset/milano_add/milano_8440.csv',
    'Milano-8815': '../dataset/milano_add/milano_8815.csv',
    'Milano-9892': '../dataset/milano_add/milano_9892.csv',
    'Milano-9924': '../dataset/milano_add/milano_9924.csv',
    # 추가 예: 'Milano-{K}': '../dataset/milano_{K}.csv',
}

# Granularity-based STL period hint
PERIOD_HINTS = {
    'ETTh1': 24, 'ETTh2': 24, 'ETTm1': 288, 'ETTm2': 288,
    'Traffic': 24, 'Electricity': 24, 'Exchange': 7,
    'Weather': 144, 'ILI': 52,
    'Milano-6165': 144,
    'Milano-8815': 144,
    'Milano-410': 144,
    'Milano-625': 144,
    'Milano-821': 144,
    'Milano-2943': 144,
    'Milano-4641': 144,
    'Milano-5163': 144,
    'Milano-5827': 144,
    'Milano-6400': 144,
    'Milano-6610': 144,
    'Milano-6710': 144,
    'Milano-6797': 144,
    'Milano-7087': 144,
    'Milano-7108': 144,
    'Milano-7315': 144,
    'Milano-8284': 144,
    'Milano-8293': 144,
    'Milano-8440': 144,
    'Milano-8815': 144,
    'Milano-9892': 144,
    'Milano-9924': 144,
}

# 이미 추출된 dataset 은 캐시(csv)에서 그대로 재사용. True 면 강제 재계산
FORCE_RECOMPUTE = False

# ----- milano_top_k 폴더 자동 등록 -----
# 아래 폴더의 milano_*.csv 를 자동 발견해 DATASET_PATHS / PERIOD_HINTS 에 추가.
# 새 Milano 지역은 이 폴더에 milano_<K>.csv 만 넣으면 자동 포함 (수동 등록 불필요).
import glob as _glob, re as _re
MILANO_AUTO_DIRS   = ['../dataset/milano_top_k']   # 스캔할 폴더 (여러 개 가능)
MILANO_PERIOD_HINT = 144                            # 자동 등록 Milano 의 STL period 기본값
_auto_added = []
for _mdir in MILANO_AUTO_DIRS:
    for _fp in sorted(_glob.glob(f'{_mdir}/milano_*.csv')):
        _mm = _re.search(r'milano[_-](\d+)', Path(_fp).stem, _re.I)
        if not _mm:
            continue
        _name = f'Milano-{_mm.group(1)}'
        if _name not in DATASET_PATHS:
            DATASET_PATHS[_name] = _fp
            _auto_added.append(_name)
        PERIOD_HINTS.setdefault(_name, MILANO_PERIOD_HINT)
print(f'Auto-registered {len(_auto_added)} Milano files from {MILANO_AUTO_DIRS}: {_auto_added}')
# =======================


def identify_ot_column(df: pd.DataFrame) -> int:
    """'OT' 컬럼 우선, 없으면 마지막 컬럼."""
    if 'OT' in df.columns:
        return int(df.columns.get_loc('OT'))
    return df.shape[1] - 1


def load_dataset(name: str, path: str) -> dict:
    df = pd.read_csv(path)
    if df.columns[0].lower() in ('date', 'time', 'timestamp'):
        df = df.drop(columns=[df.columns[0]])
    df = df.ffill().fillna(0)
    ot_idx = identify_ot_column(df)
    return {
        'name': name,
        'X': df.values.astype(np.float64),
        'ot_idx': ot_idx,
        'ot_name': str(df.columns[ot_idx]),
        'column_names': list(df.columns),
        'T': int(df.shape[0]),
        'C': int(df.shape[1]),
    }


datasets = {name: load_dataset(name, path) for name, path in DATASET_PATHS.items()}

meta_rows = [{
    'dataset': d['name'], 'T': d['T'], 'C': d['C'],
    'ot_idx': d['ot_idx'], 'ot_name': d['ot_name'],
    'period_hint': PERIOD_HINTS[d['name']],
} for d in datasets.values()]
meta_df = pd.DataFrame(meta_rows)

# dataset_metadata.csv 캐싱: 이미 있는 dataset 은 유지, 신규만 추가
meta_path = OUT_DIR / 'dataset_metadata.csv'
if meta_path.exists() and not FORCE_RECOMPUTE:
    cached_meta = pd.read_csv(meta_path)
    new_meta = meta_df[~meta_df['dataset'].isin(cached_meta['dataset'])]
    meta_df = pd.concat([cached_meta, new_meta], ignore_index=True)
meta_df.to_csv(meta_path, index=False)
meta_df

## Section 2 — 메타피처 함수 (그룹 A/B/C/D/E)

모든 함수는 OT 채널 중심. A/B/C/E는 `X_OT` 단변량, D는 `(X, ot_idx)` 다채널.
오류 시 NaN 반환 (NB2에서 imputation).

In [ ]:
import nolds
import antropy
import pycatch22
import pywt
from scipy.signal import periodogram
from scipy.spatial.distance import cosine
from scipy.stats import entropy as sp_entropy
from statsmodels.tsa.stattools import acf, pacf, adfuller, kpss, grangercausalitytests
from statsmodels.tsa.seasonal import STL
from sklearn.feature_selection import mutual_info_regression

try:
    import EntropyHub as EH
    HAS_EH = True
except Exception:
    HAS_EH = False
    print('Warning: EntropyHub not available, mse_*_ot will be NaN')


def _safe(fn, *args, **kwargs):
    """Run fn; return np.nan on any exception."""
    try:
        return fn(*args, **kwargs)
    except Exception:
        return np.nan

In [ ]:
# Group A: OT 복잡도 (단변량) — 6 features

def compute_group_a(x: np.ndarray) -> dict:
    feats = {
        'dfa_ot':              _safe(nolds.dfa, x),
        'sample_entropy_ot':   _safe(antropy.sample_entropy, x, order=2),
        'perm_entropy_ot':     _safe(antropy.perm_entropy, x, order=3, normalize=True),
        'spectral_entropy_ot': _safe(antropy.spectral_entropy, x, sf=1.0, method='welch', normalize=True),
    }
    try:
        mob, comp = antropy.hjorth_params(x)
        feats['hjorth_mobility_ot']   = float(mob)
        feats['hjorth_complexity_ot'] = float(comp)
    except Exception:
        feats['hjorth_mobility_ot']   = np.nan
        feats['hjorth_complexity_ot'] = np.nan
    return feats

In [ ]:
# Group B: OT 메모리/스케일 — 9 features (peeled into B + meta)

def _acf_first_zero(x: np.ndarray) -> float:
    nlags = min(len(x) // 4, 500)
    vals = acf(x, nlags=nlags, fft=True)
    for i, v in enumerate(vals):
        if v <= 0:
            return float(i)
    return float(len(vals))


def _pacf_sig_lags(x: np.ndarray) -> float:
    nlags = min(40, len(x) // 4)
    vals = pacf(x, nlags=nlags)
    thr = 1.96 / np.sqrt(len(x))
    return float(np.sum(np.abs(vals[1:]) > thr))


def _dominant_period(x: np.ndarray) -> float:
    f, P = periodogram(x, fs=1.0)
    if len(P) <= 1 or f[1] == 0:
        return np.nan
    idx = int(np.argmax(P[1:])) + 1
    return float(1.0 / f[idx]) if f[idx] > 0 else np.nan


def _mse_stats(x: np.ndarray) -> tuple:
    if not HAS_EH:
        return (np.nan, np.nan, np.nan)
    try:
        vals, _ = EH.MSEn(x, EH.MSobject('SampEn'), Scales=5)
        vals = np.asarray(vals, dtype=float)
        slope = float(np.polyfit(np.arange(1, len(vals) + 1), vals, 1)[0])
        return (float(np.nanmean(vals)), float(np.nanstd(vals)), slope)
    except Exception:
        return (np.nan, np.nan, np.nan)


def compute_group_b(x: np.ndarray, T: int, C: int, hl_ratio: float = 1.0) -> dict:
    mse_mean, mse_std, mse_slope = _mse_stats(x)
    return {
        'acf_first_zero_ot':  _safe(_acf_first_zero, x),
        'pacf_sig_lags_ot':   _safe(_pacf_sig_lags, x),
        'dominant_period_ot': _safe(_dominant_period, x),
        'mse_mean_ot':  mse_mean,
        'mse_std_ot':   mse_std,
        'mse_slope_ot': mse_slope,
        'length_T':     float(T),
        'n_channels_C': float(C),
        'hl_ratio':     float(hl_ratio),
    }

In [ ]:
# Group C: OT 분해/주파수 — 6 + Catch22 22 features

def _stl_strengths(x: np.ndarray, period: int) -> tuple:
    period = max(2, int(period))
    if len(x) < 2 * period:
        return (np.nan, np.nan)
    res = STL(x, period=period, robust=True).fit()
    T, S, R = np.asarray(res.trend), np.asarray(res.seasonal), np.asarray(res.resid)
    tr = max(0.0, 1.0 - np.var(R) / (np.var(T + R) + 1e-12))
    ss = max(0.0, 1.0 - np.var(R) / (np.var(S + R) + 1e-12))
    return (float(tr), float(ss))


def _adf_kpss(x: np.ndarray) -> tuple:
    try:
        adf_stat, adf_p, *_ = adfuller(x, autolag='AIC')
    except Exception:
        adf_stat, adf_p = np.nan, np.nan
    try:
        kpss_stat, kpss_p, *_ = kpss(x, regression='c', nlags='auto')
    except Exception:
        kpss_stat, kpss_p = np.nan, np.nan
    return (float(adf_stat), float(adf_p), float(kpss_stat), float(kpss_p))


def _spectral_concentration(x: np.ndarray, K: int = 5) -> float:
    _, P = periodogram(x)
    if P.sum() <= 0:
        return np.nan
    return float(np.sort(P)[-K:].sum() / P.sum())


def _wavelet_entropy(x: np.ndarray, max_scale: int = 32) -> float:
    # Downsample if too long to keep cwt fast
    xs = x[:: max(1, len(x) // 4000)]
    coeffs, _ = pywt.cwt(xs, scales=np.arange(1, max_scale), wavelet='morl')
    energy = np.mean(coeffs ** 2, axis=1)
    p = energy / (energy.sum() + 1e-12)
    return float(-np.sum(p * np.log(p + 1e-10)))


def compute_group_c(x: np.ndarray, period: int) -> dict:
    tr, ss = _safe(_stl_strengths, x, period) or (np.nan, np.nan)
    if not isinstance(tr, float):
        tr, ss = np.nan, np.nan
    adf_stat, adf_p, kpss_stat, kpss_p = _adf_kpss(x)
    return {
        'trend_strength_ot':       tr,
        'season_strength_ot':      ss,
        'adf_stat_ot':             adf_stat,
        'adf_pvalue_ot':           adf_p,
        'kpss_stat_ot':            kpss_stat,
        'kpss_pvalue_ot':          kpss_p,
        'spectral_conc_top5_ot':   _safe(_spectral_concentration, x, 5),
        'wavelet_entropy_ot':      _safe(_wavelet_entropy, x),
    }


def compute_catch22_ot(x: np.ndarray) -> dict:
    try:
        r = pycatch22.catch22_all(x)
        return {f'c22_{n}_ot': float(v) for n, v in zip(r['names'], r['values'])}
    except Exception:
        return {}

In [ ]:
# Group D: OT-vs-Others (전체 채널 사용, OT는 항상 포함) — 10 features

def compute_group_d(X: np.ndarray, ot_idx: int,
                    time_window: Optional[int] = None,
                    granger_max_lag: int = 4) -> dict:
    # Heavy datasets: shrink time dim only, never channels.
    if time_window is not None and X.shape[0] > time_window:
        X = X[:time_window]

    x_ot = X[:, ot_idx]
    others = [j for j in range(X.shape[1]) if j != ot_idx]

    # Pearson correlation
    corrs = np.array([np.corrcoef(x_ot, X[:, j])[0, 1] for j in others])
    abs_corrs = np.abs(corrs[~np.isnan(corrs)])
    pearson = {
        'corr_ot_others_mean':      float(np.mean(abs_corrs))      if abs_corrs.size else np.nan,
        'corr_ot_others_max':       float(np.max(abs_corrs))       if abs_corrs.size else np.nan,
        'corr_ot_others_top3_mean': float(np.mean(np.sort(abs_corrs)[-3:])) if abs_corrs.size else np.nan,
    }

    # Catch22 similarity
    try:
        F_ot = np.asarray(pycatch22.catch22_all(x_ot)['values'], dtype=float)
        sims = []
        for j in others:
            F_j = np.asarray(pycatch22.catch22_all(X[:, j])['values'], dtype=float)
            mask = ~(np.isnan(F_ot) | np.isnan(F_j))
            if mask.sum() >= 5:
                sims.append(1.0 - cosine(F_ot[mask], F_j[mask]))
        c22 = {
            'c22_sim_ot_others_mean': float(np.mean(sims)) if sims else np.nan,
            'c22_sim_ot_others_max':  float(np.max(sims))  if sims else np.nan,
        }
    except Exception:
        c22 = {'c22_sim_ot_others_mean': np.nan, 'c22_sim_ot_others_max': np.nan}

    # Mutual information (vectorized across channels)
    try:
        mi_vals = mutual_info_regression(X[:, others], x_ot, random_state=0)
        mi = {
            'mi_ot_others_mean': float(np.mean(mi_vals)),
            'mi_ot_others_max':  float(np.max(mi_vals)),
        }
    except Exception:
        mi = {'mi_ot_others_mean': np.nan, 'mi_ot_others_max': np.nan}

    # Granger causality ratio: fraction of channels j where j -> OT is significant
    n_causal = 0
    n_tested = 0
    for j in others:
        try:
            data = np.column_stack([x_ot, X[:, j]])  # [target, predictor]
            res = grangercausalitytests(data, maxlag=granger_max_lag, verbose=False)
            min_p = min(res[lag][0]['ssr_ftest'][1] for lag in range(1, granger_max_lag + 1))
            n_tested += 1
            if min_p < 0.05:
                n_causal += 1
        except Exception:
            continue
    granger = {'granger_ot_caused_ratio': (n_causal / n_tested) if n_tested else np.nan}

    # Variance comparison
    var_ot = float(np.var(x_ot))
    var_others = np.array([np.var(X[:, j]) for j in others])
    var_stats = {
        'ot_var_ratio':  var_ot / (np.mean(var_others) + 1e-12),
        'ot_var_zscore': (var_ot - np.mean(var_others)) / (np.std(var_others) + 1e-8),
    }

    return {**pearson, **c22, **mi, **granger, **var_stats}

In [ ]:
# Group E: OT 분포/Shift — 2 features

def _shifting(x: np.ndarray, n_windows: int = 10, n_bins: int = 20) -> float:
    windows = np.array_split(x, n_windows)
    kls = []
    for i in range(len(windows) - 1):
        h1, edges = np.histogram(windows[i], bins=n_bins, density=True)
        h2, _ = np.histogram(windows[i + 1], bins=edges, density=True)
        h1 = h1 + 1e-10
        h2 = h2 + 1e-10
        kls.append(float(sp_entropy(h1, h2)))
    return float(np.mean(kls)) if kls else np.nan


def _transition(x: np.ndarray, n_symbols: int = 3, downsample: int = 10) -> float:
    xs = x[::downsample] if len(x) > 1000 else x
    bins = np.quantile(xs, np.linspace(0, 1, n_symbols + 1))
    symbols = np.digitize(xs, bins[1:-1])
    M = np.zeros((n_symbols, n_symbols))
    for i in range(len(symbols) - 1):
        M[symbols[i], symbols[i + 1]] += 1
    M = M / (M.sum(axis=1, keepdims=True) + 1e-10)
    return float(np.trace(np.cov(M)))


def compute_group_e(x: np.ndarray) -> dict:
    return {
        'shifting_ot':   _safe(_shifting, x),
        'transition_ot': _safe(_transition, x),
    }

In [ ]:
# Top-level extractor — combines all groups
HEAVY_TIME_WINDOW = 2000   # Group D time-shrink threshold for many-channel datasets
HEAVY_CHANNEL_THR = 100


def extract_full_features(info: dict, period_hint: int) -> dict:
    X = info['X']
    ot_idx = info['ot_idx']
    x_ot = X[:, ot_idx]

    feats: dict = {'dataset': info['name']}
    feats.update(compute_group_a(x_ot))
    feats.update(compute_group_b(x_ot, info['T'], info['C']))
    feats.update(compute_group_c(x_ot, period_hint))
    feats.update(compute_catch22_ot(x_ot))
    heavy = info['C'] > HEAVY_CHANNEL_THR
    feats.update(compute_group_d(
        X, ot_idx,
        time_window=HEAVY_TIME_WINDOW if heavy else None,
    ))
    feats.update(compute_group_e(x_ot))
    return feats

## Section 3 — Full-mode 추출

In [ ]:
# Full-mode 추출 (캐시 활용: 이미 csv 에 있는 dataset 은 skip)
cache_path = OUT_DIR / 'meta_features_full.csv'
if cache_path.exists() and not FORCE_RECOMPUTE:
    cached_full = pd.read_csv(cache_path).set_index('dataset')
    existing = set(cached_full.index)
else:
    cached_full, existing = None, set()

todo = {n: info for n, info in datasets.items() if n not in existing}
print(f'cached: {sorted(existing)}  |  todo: {list(todo)}')

if todo:
    new_rows = [extract_full_features(info, period_hint=PERIOD_HINTS[name])
                for name, info in tqdm(todo.items(), desc='Full features')]
    new_df = pd.DataFrame(new_rows).set_index('dataset')
    full_df = pd.concat([cached_full, new_df]) if cached_full is not None else new_df
else:
    full_df = cached_full

full_df.to_csv(cache_path)
print('Shape:', full_df.shape)
print('NaN ratio per column (top 10):')
print(full_df.isna().mean().sort_values(ascending=False).head(10))
full_df.head()

## Section 4 — Feature Groups 매핑 (feature_groups.csv)

각 feature → (group A/B/C/D/E/meta, ot_relation self/relation/meta). NB2/NB3에서 SHAP grouping에 사용.

In [ ]:
GROUP_A = {'dfa_ot', 'sample_entropy_ot', 'perm_entropy_ot', 'spectral_entropy_ot',
           'hjorth_mobility_ot', 'hjorth_complexity_ot'}
GROUP_B_SELF = {'acf_first_zero_ot', 'pacf_sig_lags_ot', 'dominant_period_ot',
                'mse_mean_ot', 'mse_std_ot', 'mse_slope_ot'}
GROUP_B_META = {'length_T', 'n_channels_C', 'hl_ratio'}
GROUP_C_SELF = {'trend_strength_ot', 'season_strength_ot',
                'adf_stat_ot', 'adf_pvalue_ot', 'kpss_stat_ot', 'kpss_pvalue_ot',
                'spectral_conc_top5_ot', 'wavelet_entropy_ot'}
GROUP_D = {'corr_ot_others_mean', 'corr_ot_others_max', 'corr_ot_others_top3_mean',
           'c22_sim_ot_others_mean', 'c22_sim_ot_others_max',
           'mi_ot_others_mean', 'mi_ot_others_max',
           'granger_ot_caused_ratio', 'ot_var_ratio', 'ot_var_zscore'}
GROUP_E = {'shifting_ot', 'transition_ot'}


def classify(feat: str) -> tuple:
    if feat in GROUP_A:        return ('A', 'self')
    if feat in GROUP_B_SELF:   return ('B', 'self')
    if feat in GROUP_B_META:   return ('B', 'meta')
    if feat in GROUP_C_SELF:   return ('C', 'self')
    if feat.startswith('c22_') and feat.endswith('_ot'): return ('C', 'self')
    if feat in GROUP_D:        return ('D', 'relation')
    if feat in GROUP_E:        return ('E', 'self')
    return ('?', '?')


feature_groups = pd.DataFrame(
    [{'feature': f, 'group': g, 'ot_relation': r} for f, (g, r) in ((c, classify(c)) for c in full_df.columns)]
)
feature_groups.to_csv(OUT_DIR / 'feature_groups.csv', index=False)
print(feature_groups['group'].value_counts())
feature_groups.head(20)

## Section 5 — Stratified Time Sampling

In [ ]:
def stratified_time_sample(X: np.ndarray, K: int = 5, W: int = 1000, seed: int = 0) -> np.ndarray:
    T = X.shape[0]
    if T <= K * W:
        return X
    rng = np.random.default_rng(seed)
    seg = T // K
    out = []
    for k in range(K):
        s0 = k * seg
        s1 = (k + 1) * seg if k < K - 1 else T
        if s1 - s0 <= W:
            out.append(X[s0:s1])
        else:
            start = int(rng.integers(s0, s1 - W))
            out.append(X[start:start + W])
    return np.concatenate(out, axis=0)


# Per-dataset (K, W) — small datasets need smaller K/W
SAMPLE_KW = {name: (5, 1000) for name in DATASET_PATHS}
SAMPLE_KW['ILI'] = (3, 200)

## Section 6 — Sample-mode (50 repeats)

In [ ]:
N_REPEATS = 50

# Sample-mode 캐싱: long csv 에 dataset 이 이미 있으면 skip, 신규만 추가
long_path = OUT_DIR / 'meta_features_sample_long.csv'
if long_path.exists() and not FORCE_RECOMPUTE:
    cached_long = pd.read_csv(long_path)
    existing_s = set(cached_long['dataset'].unique())
else:
    cached_long, existing_s = None, set()

todo_s = {n: info for n, info in datasets.items() if n not in existing_s}
print(f'sample cached: {sorted(existing_s)}  |  todo: {list(todo_s)}')

new_long_rows = []
for name, info in tqdm(todo_s.items(), desc='Sample features'):
    K, W = SAMPLE_KW[name]
    for rep in range(N_REPEATS):
        X_s = stratified_time_sample(info['X'], K=K, W=W, seed=rep)
        sub = {**info, 'X': X_s, 'T': X_s.shape[0]}
        feats = extract_full_features(sub, period_hint=PERIOD_HINTS[name])
        for k, v in feats.items():
            if k == 'dataset':
                continue
            new_long_rows.append({'dataset': name, 'repeat': rep, 'feature': k, 'value': v})

new_long = pd.DataFrame(new_long_rows)
sample_long = pd.concat([cached_long, new_long], ignore_index=True) if cached_long is not None else new_long
sample_long.to_csv(long_path, index=False)
print('Sample long shape:', sample_long.shape)

sample_summary = (
    sample_long.groupby(['dataset', 'feature'])['value']
    .agg(['mean', 'std', 'min', 'max'])
    .reset_index()
)
sample_summary.to_csv(OUT_DIR / 'meta_features_sample_summary.csv', index=False)
sample_summary.head(10)

## Section 7 — Representativeness 검증

Sample 50회가 full을 representative 하는지 확인: z-score, KS-self-consistency, vector cosine similarity.

In [ ]:
from scipy.stats import ks_2samp

def repr_row(full_val: float, samples: np.ndarray) -> dict:
    samples = samples[~np.isnan(samples)]
    if samples.size < 2:
        return {'sample_mean': np.nan, 'sample_std': np.nan,
                'is_full_in_ci95': False, 'z_score': np.nan,
                'ks_p_value': np.nan}
    mu, sd = float(samples.mean()), float(samples.std())
    lo, hi = np.percentile(samples, [2.5, 97.5])
    odd, even = samples[::2], samples[1::2]
    _, p = ks_2samp(odd, even) if odd.size and even.size else (np.nan, np.nan)
    return {
        'sample_mean': mu, 'sample_std': sd,
        'is_full_in_ci95': bool(lo <= full_val <= hi) if not np.isnan(full_val) else False,
        'z_score': abs(full_val - mu) / (sd + 1e-8) if not np.isnan(full_val) else np.nan,
        'ks_p_value': float(p) if not np.isnan(p) else np.nan,
    }


feat_cols = [c for c in full_df.columns]
rep_rows = []
for name in DATASET_PATHS:
    sub = sample_long[sample_long['dataset'] == name]
    full_vec = full_df.loc[name]
    sample_vecs = sub.pivot(index='repeat', columns='feature', values='value')
    sample_vecs = sample_vecs.reindex(columns=feat_cols)
    # Per-feature stats
    for feat in feat_cols:
        vals = sample_vecs[feat].to_numpy()
        row = {'dataset': name, 'feature': feat, 'full_value': float(full_vec[feat]) if pd.notna(full_vec[feat]) else np.nan}
        row.update(repr_row(full_vec[feat], vals))
        rep_rows.append(row)
    # Vector cosine (full vs each repeat) — store as separate file? Skip per-dataset rows; merge into report.

representativeness = pd.DataFrame(rep_rows)
representativeness.to_csv(OUT_DIR / 'representativeness_report.csv', index=False)

# Quick summary
print('Median z-score (per dataset):')
print(representativeness.groupby('dataset')['z_score'].median())
representativeness.head(10)

## Section 8 — 시각화 (inline only)

In [ ]:
# Figure 1: per-dataset feature distribution (sample box + full dot) for 8 key features
KEY = ['dfa_ot', 'sample_entropy_ot', 'trend_strength_ot', 'season_strength_ot',
       'corr_ot_others_max', 'mi_ot_others_mean', 'granger_ot_caused_ratio', 'n_channels_C']
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
for ax, feat in zip(axes.flatten(), KEY):
    if feat not in sample_long['feature'].unique():
        ax.set_visible(False); continue
    sub = sample_long[sample_long['feature'] == feat]
    order = list(DATASET_PATHS)
    sns.boxplot(data=sub, x='dataset', y='value', order=order, ax=ax, color='lightsteelblue')
    for i, name in enumerate(order):
        v = full_df.loc[name, feat] if feat in full_df.columns else np.nan
        if pd.notna(v):
            ax.scatter(i, v, color='crimson', zorder=5, s=40, marker='D', label='full' if i == 0 else None)
    ax.set_title(feat); ax.tick_params(axis='x', rotation=45)
plt.suptitle('Sample (box) vs Full (red diamond) per feature', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Figure 2: PCA of meta-features (10 datasets in 2D)
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X = full_df.fillna(full_df.median(numeric_only=True))
Xs = StandardScaler().fit_transform(X.values)
pca = PCA(n_components=2, random_state=42).fit(Xs)
Z = pca.transform(Xs)

plt.figure(figsize=(7, 6))
plt.scatter(Z[:, 0], Z[:, 1], s=120, c='steelblue', edgecolor='black')
for i, name in enumerate(X.index):
    plt.annotate(name, (Z[i, 0], Z[i, 1]), xytext=(6, 4), textcoords='offset points')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%})')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%})')
plt.title('PCA of OT-centric meta-features (10 datasets)')
plt.tight_layout(); plt.show()

In [ ]:
# Figure 3: representativeness heatmap (dataset × feature → z-score, top-30 features)
top_feats = (representativeness.groupby('feature')['z_score']
             .median().sort_values(ascending=False).head(30).index.tolist())
pivot = representativeness[representativeness['feature'].isin(top_feats)].pivot(
    index='dataset', columns='feature', values='z_score')
plt.figure(figsize=(16, 5))
sns.heatmap(pivot, cmap='YlOrRd', annot=False, cbar_kws={'label': 'z-score (full vs sample)'})
plt.title('Representativeness z-score (top-30 features by median z)')
plt.tight_layout(); plt.show()